Nom : Khadijetou Mohamed abe

Filière : GLCC

# 1. CODE SCRATCH

In [1]:
import numpy as np
import random


def choisir_action(s, Q, epsilon):
    # """
    # Exploration vs Exploitation
    # Permet à l'agent de choisir entre découvrir de nouvelles cases ou utiliser ce qu'il sait.
    # """
    # random.uniform(0, 1) tire un chiffre à virgule entre 0 et 1.
    if random.uniform(0, 1) < epsilon:
        # EXPLORATION : On choisit une action au hasard (0:Haut, 1:Droite, 2:Bas, 3:Gauche)
        return random.choice([0, 1, 2, 3])
    else:
        # EXPLOITATION : On prend l'action qui a la plus grande valeur dans la table Q pour l'état s
        # np.argmax va chercher l'indice de la colonne qui contient le plus grand score pour la ligne 's'
        return np.argmax(Q[s, :])



def step_env(state, action):
    # """
    # Calcule le prochain état et la récompense selon l'action choisie.
    # Grille 5x5 : 25 états (0 à 24).
    # Les murs sont impassables, l'agent reçoit une punition s'il les touche.
    # """
    # On transforme l'état (un chiffre de 0 à 24) en coordonnées matricielles X, Y.
    # La division entière (// 5) donne la ligne, le modulo (% 5) donne la colonne.
    row = state // 5 
    col = state % 5
    
    # On sauvegarde l'état initial avant mouvement au cas où le mouvement serait impossible
    old_state = state
    
    # On tente de modifier les coordonnées selon l'action
    if action == 0:   # Haut (on monte d'une ligne)
        row -= 1
    elif action == 1: # Droite (on avance d'une colonne)
        col += 1
    elif action == 2: # Bas (on descend d'une ligne)
        row += 1
    elif action == 3: # Gauche (on recule d'une colonne)
        col -= 1
        
    # Vérification 1 : 
    # Si la ligne ou la colonne devient négative ou dépasse 4, on sort de la grille 5x5.
    if row < 0 or row > 4 or col < 0 or col > 4:
        # on annule le mouvement (il reste sur old_state)
        # On lui donne une pénalité de -2 pour qu'il apprenne à ne pas recommencer
        return old_state, -2, False
        
    # On calcule le numéro de la nouvelle case potentielle
    next_state = row * 5 + col
    
    # Vérification 2 :
    # On vérifie si la nouvelle case fait partie de notre liste globale de murs
    if next_state in murs:
        # il reste sur sa case d'origine (old_state)
        # On lui donne une pénalité de -2
        return old_state, -2, False 
        
    # Vérification 3 : 
    if next_state == 24:
        # 24 est la Sortie 
        # On donne la récompense maximale +10 et on termine le jeu (done = True)
        return next_state, 10, True 
        
    # Sinon, mouvement normal réussi (case vide)
    # On donne -1 pour l'encourager à trouver le chemin le plus court possible
    return next_state, -1, False 


def q_learning_scratch(episodes=500, alpha=0.1, gamma=0.9, epsilon=0.2):

    # Initialisation de la table Q avec 25 états et 4 actions
    # Les lignes représentent les états (les 25 cases de la grille).
    # Les colonnes représentent les actions possibles (Haut, Droite, Bas, Gauche).
    Q =  np.zeros((25, 4))
    
    for k in range(episodes):
        # Remet l'agent à la case de départ au début de chaque épisode.
        # L'état 0 correspond à la case en haut à gauche de la matrice.
        s = 0
    
        done = False    
        while not done:
            # L'agent choisit une action (avec une part de hasard définie par epsilon)
            a = choisir_action(s, Q, epsilon)
            
            # L'environnement renvoie le nouvel état et la récompense
            s_prime, reward, done = step_env(s, a)
            
            # '''
            # Calcul de la Target pour mettre à jour la valeur Q(s,a)
            # Formule : Target = Récompense + Gamma * Max( Q(état_suivant) )
            # '''
            if done:
                # Si l'état est terminal (Sortie), il n'y a pas d'état suivant.
                target = reward
            else:
                # Sinon, on anticipe la meilleure récompense possible depuis la case suivante
                target = reward + gamma * np.max(Q[s_prime, :])
            
            # '''
            # Mise à jour de la table Q 
            # Formule : Q(s,a) = Q(s,a) + Alpha * (Target - Q(s,a))
            # '''
            Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])
            
            # On remplace l'état courant par le nouvel état pour l'itération suivante
            s = s_prime
            
    return Q

## a. Environnement

In [2]:
murs = [1, 2, 3, 7, 11, 18, 20, 23]

def afficher_matrice(agent_state, step_num):
    # """
    # Affiche la grille 5x5 étape par étape.
    # [A] = Agent 
    # [S] = Sortie (indice 24)
    # [#] = Murs
    # [ ] = Case vide
    # """
    print(" Étape ",step_num)
    
    for r in range(5):
        ligne_affichage = ""
        for c in range(5):
            etat_courant = r * 5 + c
            
            if etat_courant == agent_state:
                ligne_affichage += "[A] " 
            elif etat_courant == 24:
                ligne_affichage += "[S] " 
            elif etat_courant in murs:
                ligne_affichage += "[#] " 
            else:
                ligne_affichage += "[ ] " 
        print(ligne_affichage)


## b. Exécution du code scratch

In [3]:
# Entraînement de l'agent
Q_table_optimale = q_learning_scratch(episodes=2000, alpha=0.1, gamma=0.9, epsilon=0.1)


## c. Affichage

In [5]:
etat_actuel = 0
done = False
etape = 0

afficher_matrice(etat_actuel, etape)
print("\n")

noms_actions = {0: "Haut", 1: "Droite", 2: "Bas", 3: "Gauche"}

while not done:
    etape += 1
    
    # Exploitation (epsilon=0) : l'agent utilise sa mémoire (table Q)
    meilleure_action = np.argmax(Q_table_optimale[etat_actuel, :])
    
    print(f">> L'agent décide d'aller vers : {noms_actions[meilleure_action]}")
    
    # L'agent bouge et on récupère la récompense de l'environnement
    etat_suivant, recompense, done = step_env(etat_actuel, meilleure_action)
    
    etat_actuel = etat_suivant
    afficher_matrice(etat_actuel, etape)
    print("\n")

print("Mission accomplie ! L'agent a trouvé son chemin dans le labyrinthe.")

 Étape  0
[A] [#] [#] [#] [ ] 
[ ] [ ] [#] [ ] [ ] 
[ ] [#] [ ] [ ] [ ] 
[ ] [ ] [ ] [#] [ ] 
[#] [ ] [ ] [#] [S] 


>> L'agent décide d'aller vers : Bas
 Étape  1
[ ] [#] [#] [#] [ ] 
[A] [ ] [#] [ ] [ ] 
[ ] [#] [ ] [ ] [ ] 
[ ] [ ] [ ] [#] [ ] 
[#] [ ] [ ] [#] [S] 


>> L'agent décide d'aller vers : Bas
 Étape  2
[ ] [#] [#] [#] [ ] 
[ ] [ ] [#] [ ] [ ] 
[A] [#] [ ] [ ] [ ] 
[ ] [ ] [ ] [#] [ ] 
[#] [ ] [ ] [#] [S] 


>> L'agent décide d'aller vers : Bas
 Étape  3
[ ] [#] [#] [#] [ ] 
[ ] [ ] [#] [ ] [ ] 
[ ] [#] [ ] [ ] [ ] 
[A] [ ] [ ] [#] [ ] 
[#] [ ] [ ] [#] [S] 


>> L'agent décide d'aller vers : Droite
 Étape  4
[ ] [#] [#] [#] [ ] 
[ ] [ ] [#] [ ] [ ] 
[ ] [#] [ ] [ ] [ ] 
[ ] [A] [ ] [#] [ ] 
[#] [ ] [ ] [#] [S] 


>> L'agent décide d'aller vers : Droite
 Étape  5
[ ] [#] [#] [#] [ ] 
[ ] [ ] [#] [ ] [ ] 
[ ] [#] [ ] [ ] [ ] 
[ ] [ ] [A] [#] [ ] 
[#] [ ] [ ] [#] [S] 


>> L'agent décide d'aller vers : Haut
 Étape  6
[ ] [#] [#] [#] [ ] 
[ ] [ ] [#] [ ] [ ] 
[ ] [#] [A] [ ] [